## S-curve (logistic) technology adoption functions

Helper functions for `Mitigation_AllowableEmissions_CDR.ipynb` (section 4), loaded there with `%run nemet_functions.ipynb`.

Based on Nemet, G., Greene, J., Müller-Hansen, F., & Minx, J. C. (2023). Dataset on the adoption of historical technologies informs the scale-up of emerging carbon dioxide removal measures. *Communications Earth & Environment*, 4, 397. https://doi.org/10.1038/s43247-023-01056-1

- `logistic`: logistic function (Nemet Eq. 1)
- `B_from_k`, `k_from_B`: convert between the logistic coefficient $k$ (metric A) and growth around the inflection point (metric B)
- `inflection_from_start`: inflection year for a curve passing through a given starting value
- `nemet_metrics`: metrics A, B, C of Nemet Fig. 1 and key years
- `plot_adoption_scurve`: three-panel plot in the style of Nemet Fig. 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Helper functions for S-curve (logistic) technology adoption, following Nemet et al. (2023)

def logistic(t, L, k, t0):
    """Logistic function, Nemet et al. Eq. 1. L = asymptote, k = growth coefficient (1/yr), t0 = inflection year"""
    return L / (1 + np.exp(-k * (np.asarray(t, dtype=float) - t0)))

def B_from_k(k):
    """Nemet metric B (fraction/yr): least-squares linear slope of the logistic from t0-5 to t0+5,
    normalized by the value at the inflection point (L/2)"""
    dt = np.arange(-5, 6)
    f = logistic(dt, 1.0, k, 0.0)
    slope = np.sum(dt * (f - f.mean())) / np.sum(dt**2)
    return slope / 0.5

def k_from_B(B_percent):
    """Convert Nemet metric B (in %/yr) to the logistic coefficient k (in 1/yr)"""
    B = B_percent / 100
    B_max = B_from_k(1e3)
    if not 0 < B < B_max:
        raise ValueError(f"Metric B must be between 0 and {100*B_max:.0f} %/yr")
    lo, hi = np.log(1e-6), np.log(1e3)   # bisection on log(k)
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        if B_from_k(np.exp(mid)) < B:
            lo = mid
        else:
            hi = mid
    return np.exp(0.5 * (lo + hi))

def inflection_from_start(start_year, start_value, L, k):
    """Inflection year t0 such that the logistic passes through start_value in start_year"""
    return start_year + np.log(L / start_value - 1) / k

def nemet_metrics(L, k, t0, start_year):
    """Nemet et al. Fig. 1 growth metrics (in %/yr) and key years for a logistic curve"""
    L = abs(L)
    t_2p5 = t0 - np.log(39) / k                 # year when the curve reaches 2.5% of L
    if t_2p5 >= start_year + 1:                 # C: exponential fit over the formative phase (Nemet Eq. 2)
        tc = np.arange(start_year, np.floor(t_2p5) + 1)
        C = 100 * np.polyfit(tc, np.log(logistic(tc, L, k, t0)), 1)[0]
    else:
        C = np.nan
    return {'A': 100 * k, 'B': 100 * B_from_k(k), 'C': C,
            't0': t0, 't_2p5': t_2p5, 'dt_10_90': np.log(81) / k}

def plot_adoption_scurve(L, k, t0, start_year, end_year, ylabel, title):
    """Plot an S-curve with the three growth metrics of Nemet et al. (2023), Fig. 1"""
    L = abs(L)
    m = nemet_metrics(L, k, t0, start_year)
    t_last = max(end_year, t0 + 1.5 * m['dt_10_90'])
    t = np.linspace(min(start_year, t0 - 1.5 * m['dt_10_90']), t_last, 500)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    fig.suptitle(f"{title}: S-curve metrics as in Nemet et al. (2023), Fig. 1")

    # (a) logistic coefficient: asymptote and inflection point
    ax[0].plot(t, logistic(t, L, k, t0), 'k', lw=2)
    ax[0].axhline(L, color='tab:orange', lw=1)
    ax[0].text(t[0], L * 0.93, ' Asymptote L', color='tab:orange')
    ax[0].plot(t0, L / 2, 'o', color='m', ms=8)
    ax[0].text(t0, L / 2, f'  Inflection point ({t0:.0f})', color='m', va='top')
    ax[0].axvspan(start_year, end_year, color='0.9', zorder=0, label=f'this notebook ({start_year}-{end_year})')
    ax[0].set_title(f"(a) Logistic coefficient A = k = {m['A']:.1f} %/yr")
    ax[0].legend(loc='lower right', fontsize=8)

    # (b) linear growth around the inflection point
    tb = np.linspace(t0 - 10, t0 + 10, 200)
    tt = t0 + np.arange(-5, 6)
    p = np.polyfit(tt, logistic(tt, L, k, t0), 1)
    ax[1].plot(tb, logistic(tb, L, k, t0), 'k', lw=2)
    ax[1].plot(tt, np.polyval(p, tt), color='tab:orange', lw=2)
    ax[1].plot([t0 - 5, t0 + 5], logistic([t0 - 5, t0 + 5], L, k, t0), 'o', color='tab:blue', ms=8)
    ax[1].plot(t0, L / 2, 'o', color='m', ms=8)
    for x in (t0 - 5, t0, t0 + 5):
        ax[1].axvline(x, color='k', ls=':', lw=0.8)
    ax[1].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax[1].set_title(f"(b) Linear around inflection B = {m['B']:.1f} %/yr")

    # (c) exponential growth in the formative phase (log axis, so exponential growth is a straight line)
    if np.isfinite(m['C']):
        tc = np.linspace(start_year, m['t_2p5'] + 0.3 * (m['t_2p5'] - start_year), 200)
        ax[2].semilogy(tc, logistic(tc, L, k, t0), 'k', lw=2)
        ax[2].axhline(0.025 * L, color='tab:orange', lw=1)
        ax[2].text(start_year, 0.025 * L * 1.3, ' 2.5% of L', color='tab:orange')
        ax[2].axvspan(start_year, m['t_2p5'], color='0.9', zorder=0)
        ax[2].text(start_year, logistic(start_year, L, k, t0), f' Formative phase\n ({start_year}-{m["t_2p5"]:.0f})', va='bottom')
        ax[2].set_title(f"(c) Formative phase exponential C = {m['C']:.1f} %/yr")
    else:
        ax[2].text(0.5, 0.5, f'Already past the formative phase\n(2.5% of L) in {start_year}',
                   ha='center', va='center', transform=ax[2].transAxes)
        ax[2].set_title("(c) Formative phase exponential")
        ax[2].set_xticks([])
        ax[2].set_yticks([])
    for a in ax:
        a.set_xlabel('year')
        a.set_ylabel(ylabel)
        a.grid(True)
    plt.tight_layout()
    plt.show()

    print(f"Metric A (logistic coefficient k):          {m['A']:.1f} %/yr   (historical median 13)")
    print(f"Metric B (linear around inflection point):  {m['B']:.1f} %/yr   (historical median 6.2, range 1.1-14.3)")
    if np.isfinite(m['C']):
        print(f"Metric C (formative phase exponential):     {m['C']:.1f} %/yr   (historical median 13, range 1-36)")
    print(f"Formative phase ends (2.5% of asymptote):   {m['t_2p5']:.0f}")
    print(f"Inflection year (50% of asymptote):         {m['t0']:.0f}")
    print(f"Time from 10% to 90% of asymptote:          {m['dt_10_90']:.0f} years")